In [ ]:
# 在 Google Colab 里运行时，先把本书的代码拉下来（本地 Jupyter 会跳过这一格）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/pytorch-inside-out"):
        !git clone -q --depth 1 https://github.com/wikieden/pytorch-inside-out /content/pytorch-inside-out
    %cd /content/pytorch-inside-out/code/notebooks


> 在线讲义：[第 18 章 · DDP：每卡一份模型，反向时边算边对答案](https://wikieden.github.io/pytorch-inside-out/lessons/18-ddp.html) · 本书《PyTorch 从里到外》

# 第 18 章 · DDP：每卡一份模型，反向时边算边对答案

本章回答：DistributedDataParallel 包一下模型，背后做了哪三件事？为什么梯度要"分桶"发？通信怎么藏到反向计算后面？梯度累积时为什么要 `no_sync`？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/18-ddp.md`（讲解、动画、练习都在讲义里）。对应原文：Ultra-Scale Playbook §Data Parallelism（`ultrascale-playbook.md` 第 560 行起：First optimization 第 604 行、Second optimization: Bucketing 第 645 行、Third optimization: gradient accumulation 第 665 行）；AISPE ch04 §4.1.3（`ai-infra/.../chapters/ch04.md` 第 87–260 行，DDP 重叠在第 201 行起）；本章输出在本机 Gloo 2 进程上跑过

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import sys
sys.path.insert(0, "..")   # 子进程要能按名字导入函数，所以用本地模块

from torch.nn.parallel import DistributedDataParallel as DDP
import torch
import torch.nn as nn

import ch18_ddp as M          # 传给 run(...) 的函数从这里取
from dist_util import run
torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 包装时 broadcast | 起点一致 | `ddp_demo` 的 weight | 两个 rank 相同 |
| 第二步 反向时平均 | [1, 0] 与 [0, 1] | `ddp_demo` 的 grad | 都是 [0.5, 0.5] |
| 第三步 分桶重叠 | 4 段 × 10 ms，每桶 8 ms | `bucket_timeline()` | 不重叠 72 ms，重叠 48 ms |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `bucket_timeline`**（来自 `ch18_ddp.py`）：分桶重叠的总用时（不算启动开销）：返回 ms。

In [ ]:
def bucket_timeline(n_buckets=4, backward_ms=40, comm_ms=32, overlap=True):
    """分桶重叠的总用时（不算启动开销）：返回 ms。"""
    seg, per = backward_ms / n_buckets, comm_ms / n_buckets
    t = 0.0
    for b in range(n_buckets):
        start = (b + 1) * seg if overlap else backward_ms
        t = max(t, start) + per
    return max(backward_ms, t)

**函数 `ddp_demo`**（来自 `ch18_ddp.py`）：下面要用到的小工具。

In [ ]:
def ddp_demo(rank, world):
    torch.manual_seed(rank)                       # 故意让两个 rank 初始化不同
    m = nn.Linear(2, 1)
    d = DDP(m)                                    # 包装时 broadcast rank 0 的参数
    w_after = m.weight.detach().clone().tolist()
    xb = torch.tensor([[1.0, 0.0]]) if rank == 0 else torch.tensor([[0.0, 1.0]])
    d(xb).sum().backward()                        # 本地梯度 [1, 0] / [0, 1]，all_reduce 后取平均
    return dict(weight=w_after, grad=m.weight.grad.tolist())

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 DDP 三件事：")
for r, res in enumerate(run(M.ddp_demo)):
    print(f"  rank {r}: {res}")

In [ ]:
print("\n第三步 4 段反向各 10 ms、每桶通信 8 ms：算完再发", bucket_timeline(overlap=False), "ms，分桶重叠", bucket_timeline(), "ms")

## 3. 代码里才看得到的

1. **DDP 在 CPU 上不传 `device_ids`**：GPU 时才写 `device_ids=[local_rank]`。
2. **`bucket_timeline` 不算启动开销**：动画里每次通信多 1 ms，所以数字略大。

## 4. 自己改一改

- `bucket_timeline(n_buckets=1)` 和 `n_buckets=8` 比一比。
- 把 `ddp_demo` 里两个 rank 的输入都改成 [[1, 0]]，看平均梯度。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 每卡一份模型吃不同数据，梯度平均一起改；
> 包装时广播参数，反向时 all_reduce 取平均，更新各自做；
> 梯度好一桶发一桶，通信藏在计算后；累积前几步 no_sync，参数没进 loss 要报错。